In [1]:
import copy
import gc
import json
import math
import numpy as np
import os
import pandas as pd
import requests

from scipy.spatial.distance import cosine, euclidean

import IPython.core.display


In [2]:
distance_col_labels = {
    'vector_dist_normal': 'Embedding',
    'geo_dist_normal': 'Geospatial',
    'pred__24_object_type__v_dist_normal': 'Object type',
    'pred__24_vessel_form__v_dist_normal': 'Vessel form',
    'pred__24_motif__v_dist_normal': 'Motif',
    'pred__24_fabric_category__v_dist_normal': 'Fabric',
    'pred__24_decorative_technique__v_dist_normal': 'Decorative tech.',
    'pred__24_vessel_part_present__v_dist_normal': 'Vessel part',
}

distance_weights = {
    'vector_dist_normal': 0.39,
    'geo_dist_normal': 0.01,
    'pred__24_object_type__v_dist_normal': 0.0,
    'pred__24_vessel_form__v_dist_normal': 0.1,
    'pred__24_motif__v_dist_normal': 0.1,
    'pred__24_fabric_category__v_dist_normal': 0.0,
    'pred__24_decorative_technique__v_dist_normal': 0.1,
    'pred__24_vessel_part_present__v_dist_normal': 0.1
}



In [3]:
# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
pc_data_path = os.path.join(
    repo_path, 'files', 'poggio-civitate',
)
# this is the path to the main dataset, with all of the cataloged objects.
object_data_path = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings.csv',
)
output_html = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings.html',
)
MAX_RECORDS = 1500
KEYWORD = 'bucchero-diag'
TITLE = 'Poggio Civitate Cataloged Diagnostic Bucchero Sherds'
file_keyword = str(KEYWORD).replace(' ', '-').lower()
file_suffix = '-multi'

sampled_csv =  os.path.join(
    pc_data_path, f'pc-catalog-objs-sampled-{file_keyword}{file_suffix}.csv',
)


distances_csv =  os.path.join(
    pc_data_path, f'pc-catalog-objs-distance-{file_keyword}{file_suffix}.csv',
)


net_output_html = os.path.join(
    pc_data_path, f'pc-catalog-objs-embeddings-{file_keyword}{file_suffix}.html',
)


df_all = pd.read_csv(object_data_path)
print(f'We have {len(df_all.index)} records of object data')
print(df_all.columns.tolist())

We have 13541 records of object data
['Unnamed: 0', 'uuid', 'label', 'path', 'item_class__slug', 'project__label', 'project__uuid', 'geo_source__path', 'geo_source__uuid', 'chrono_source__path', 'chrono_source__uuid', 'latitude', 'longitude', 'earliest', 'latest', 'str_for_embedding', 'embedding', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'thumbnail_uri', 'pred__24_object_type', 'pred__24_vessel_form', 'pred__24_motif', 'pred__24_fabric_category', 'pred__24_decorative_technique', 'pred__24_vessel_part_present', 'pred__24_object_type__v', 'pred__24_vessel_form__v', 'pred__24_motif__v', 'pred__24_fabric_category__v', 'pred__24_decorative_technique__v', 'pred__24_vessel_part_present__v']


In [4]:
def get_uuids_by_query(url):
    if not url:
        return None
    if '#' in url:
        url = url.split('#')[0]
    if not 'response=uuid' in url:
        url += '&response=uuid'
    if not 'rows=' in url:
        url += '&rows=2000'
    headers = {
        'User-Agent': 'oc-api-client',
        'Accept': 'application/json', 
    }
    r = requests.get(url, headers=headers)
    uuids = r.json()
    return uuids

url = None 
# bucchero vessels
# url = 'https://opencontext.org/query/Europe/Italy.json?response=uuid&rows=2000&cat=oc-gen-cat-object&proj=24-murlo&prop=24-fabric-category---24-bucchero&prop=24-object-type---24-vessel&prop=24-vessel-form&type=subjects'    
# Bowls
# url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&images=1&proj=24-murlo&prop=24-object-type---24-vessel&prop=24-vessel-form---24-bowl&type=subjects#tab=3/aq=related-media---undefined/ovgrd=oc/zm=15/ov=sqr'
# Diagnostic bucchero
url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&proj=24-murlo&prop=24-fabric-category---24-bucchero&prop=24-object-type---24-vessel&prop=24-vessel-form---24-bowl%7C%7C24-cup%7C%7C24-lid%7C%7C24-plate%7C%7C24-vase%7C%7C24-kantharos%7C%7C24-oinochoe%7C%7C24-tondo%7C%7C24-dish%7C%7C24-tripod%7C%7C24-skyphos%7C%7C24-cooking-stand%7C%7C24-alabastron%7C%7C24-jar%7C%7C24-jug%7C%7C24-krater%7C%7C24-kylix%7C%7C24-lekythos%7C%7C24-omphalos&type=subjects'
# Rocchetti
# url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&proj=24-murlo&prop=24-object-type---24-textile-related---24-textile-relatedrocchetto&type=subjects'
# Spindle whorls
# url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&proj=24-murlo&prop=24-object-type---24-textile-related---24-textile-relatedspindle-whorl&type=subjects#tab=0/aq=facet-24-object-type---24-textile-relatedspindle-whorl/ovgrd=oc/zm=15/lat=43.1614/lng=11.3973/ov=sqr'
# diagnostic pottery
# url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&images=1&proj=24-murlo&prop=24-object-type---24-vessel&prop=24-vessel-form---24-bowl%7C%7C24-cup%7C%7C24-plate%7C%7C24-lid%7C%7C24-vase%7C%7C24-oinochoe%7C%7C24-pithos%7C%7C24-kantharos%7C%7C24-globular-pot%7C%7C24-other-5%7C%7C24-dish%7C%7C24-tondo%7C%7C24-amphora%7C%7C24-cooking-stand%7C%7C24-aryballos%7C%7C24-cooking-bell%7C%7C24-jug%7C%7C24-cooking-vessel%7C%7C24-jar%7C%7C24-cooking-pan-plate%7C%7C24-alabastron%7C%7C24-strainer-sieve%7C%7C24-krater%7C%7C24-kylix%7C%7C24-skyphos%7C%7C24-pyxis%7C%7C24-focolus%7C%7C24-tripod%7C%7C24-lekythos%7C%7C24-lamp%7C%7C24-stopper%7C%7C24-cauldron%7C%7C24-flask%7C%7C24-heavy-vessel%7C%7C24-omphalos&type=subjects#tab=0/aq=related-media---undefined/ovgrd=oc/zm=15/lat=43.1614/lng=11.3973/ov=sqr'
uuids = None
uuids = get_uuids_by_query(url)
if uuids:
    print(f'UUIDs to process: {len(uuids)}')

UUIDs to process: 383


In [5]:
def get_valid_data_sample(df_all, keyword=KEYWORD, uuids=None, n=MAX_RECORDS, filepath=sampled_csv):
    if filepath and os.path.exists(filepath):
        print(f'Reading previously cached sample data from {filepath}')
        df_sample = pd.read_csv(filepath)
        df_sample['embedding'] = df_sample['embedding'].apply(lambda x: json.loads(x))
        return df_sample
    valid_index = (
        ~df_all['embedding'].isnull()
        & ~df_all['latitude'].isnull()
        & ~df_all['longitude'].isnull()
        & ~df_all['uuid'].isnull()
    )
    if keyword and not uuids:
        valid_index &= df_all['str_for_embedding'].str.contains(keyword)
    if uuids and isinstance(uuids, list):
        valid_index &= df_all['uuid'].isin(uuids)
    if len(df_all[valid_index].index) < n:
        n = len(df_all[valid_index].index)
    df_sample = df_all[valid_index].sample(n=n, random_state=1).copy()
    df_sample.reset_index(drop=True, inplace=True)
    if filepath:
        df_sample.to_csv(filepath, index=False)
    df_sample['embedding'] = df_sample['embedding'].apply(lambda x: json.loads(x))
    return df_sample

df_sample = get_valid_data_sample(df_all, keyword=KEYWORD, uuids=uuids, n=MAX_RECORDS)
print(f'We have {len(df_sample.index)} valid sample records of object data')

Reading previously cached sample data from /home/ekansa/github/open-context-jupyter/files/poggio-civitate/pc-catalog-objs-sampled-bucchero-diag-multi.csv
We have 383 valid sample records of object data


In [6]:
# Release some memory
del df_all
gc.collect()

0

In [7]:
def get_difference_between_embeddings(e1, e2):
    """Get a difference distance between two embeddings"""
    v1 = np.array(e1)
    v2 = np.array(e2)
    return cosine(v1, v2)

def get_geodist(lat_1, lon_1, lat_2, lon_2, max_dist=None):
    lat_sqr = (lat_1 - lat_2)**2
    lon_sqr = (lon_1 - lon_2)**2
    dist = math.sqrt((lat_sqr + lon_sqr))
    if not max_dist:
        return dist
    # Return the distance, normalized as a fraction of the
    # max_distance
    return dist / max_dist


In [8]:
from itertools import combinations

def make_raw_distance_matrix(df_sample, filepath=distances_csv):
    if os.path.exists(filepath):
        # We have already cacheded the distance data, no need to recalcuate
        df_raw_dist = pd.read_csv(filepath)
        return df_raw_dist
    # make the maximum geo distance to normalize the geo distances
    max_lat = df_sample['latitude'].max()
    max_lon = df_sample['longitude'].max()
    min_lat = df_sample['latitude'].min()
    min_lon = df_sample['longitude'].min()
    max_geo_dist = get_geodist(lat_1=max_lat, lon_1=max_lon, lat_2=min_lat, lon_2=min_lon)
    count_sample = len(df_sample.index)

    # get cols for columns relating to controlled vocab predicates
    cvocab_cols = [c for c in df_sample.columns.tolist() if c.startswith('pred_') and c.endswith('__v')]
    print(f'Prepare arrays for: {cvocab_cols}')
    for cvocab_col in cvocab_cols:
        df_sample[cvocab_col] = df_sample[cvocab_col].apply(json.loads)
        df_sample[cvocab_col] = df_sample[cvocab_col].apply(np.array)

    df_sample['embedding'] = df_sample['embedding'].apply(np.array)
    df_sample['uuid_g'] = df_sample['uuid'].str.replace('-', '_')
    uuid_g_list = df_sample['uuid_g'].unique().tolist()
    # make unique combinations of all sources and target uuid_g
    rows = [{'source': a, 'target': b,} for a, b in combinations(uuid_g_list, 2)]
    df_raw_dist = pd.DataFrame(data=rows)

    print('generated source, target dataframe...')

    join_cols = ['uuid_g', 'uuid', 'label', 'latitude', 'longitude', 'embedding'] + cvocab_cols
    df_raw_dist = df_raw_dist.merge(df_sample[join_cols], left_on='source', right_on='uuid_g', )
    df_raw_dist = df_raw_dist.merge(df_sample[join_cols], left_on='target', right_on='uuid_g', suffixes=('_sub', '_trg'))

    df_raw_dist['geo_dist'] = np.sqrt( 
        (df_raw_dist['latitude_sub'] - df_raw_dist['latitude_trg'])**2 
        + (df_raw_dist['longitude_sub']-df_raw_dist['longitude_trg'])**2
    )
    df_raw_dist['geo_dist_normal'] = df_raw_dist['geo_dist'] / max_geo_dist

    for cvocab_col in cvocab_cols:
        col_dist = cvocab_col + '_dist'
        col_dist_norm = col_dist + '_normal'
        print(f'Calculating euclidean distance for: {col_dist}')
        df_raw_dist[col_dist] = df_raw_dist.apply(
            lambda x: euclidean(x[f'{cvocab_col}_sub'], x[f'{cvocab_col}_trg']), 
            axis=1
        )
        col_max = df_raw_dist[col_dist].max()
        df_raw_dist[col_dist_norm] = df_raw_dist[col_dist] / col_max
        
    print('finished euclidean distances...')
    df_raw_dist['vector_dist'] = df_raw_dist.apply(
        lambda x: cosine(x['embedding_sub'], x['embedding_trg']), 
        axis=1
    )
    print('finished embedding cosine similarities...')
    df_raw_dist['vector_sim'] = 1 - df_raw_dist['vector_dist']
    max_vector_dist =  df_raw_dist['vector_dist'].max()
    df_raw_dist['vector_dist_normal'] = df_raw_dist['vector_dist'] / max_vector_dist
    df_raw_dist.drop_duplicates(subset=['source', 'target'], inplace=True)

    drop_cvocb_cols = [f'{c}_sub' for c in cvocab_cols]
    drop_cvocb_cols += [f'{c}_trg' for c in cvocab_cols]
    df_raw_dist.drop(columns=(['embedding_sub', 'embedding_trg'] +  drop_cvocb_cols), inplace=True)
    df_raw_dist.to_csv(filepath, index=False)
    return df_raw_dist
            
# Make raw distance dataframe
df_raw_dist = make_raw_distance_matrix(df_sample)
# df_raw_dist.head(10)
            

In [9]:

def make_weighted_similaries(df_raw_dist, distance_weights=distance_weights):
    df_sim = df_raw_dist.copy()
    cols = df_sim.columns.tolist()
    use_distance_weights = {k:v for k, v in distance_weights.items() if k in cols}
    df_sim['distance'] = 0.0
    total_sim = len(df_sim.index)
    for col, weight in use_distance_weights.items():
        bad_index = df_sim[col].isnull()
        if not df_sim[bad_index].empty:
            print(f'{len(df_sim[bad_index].index)} of {total_sim} with null results for {col}')
            # df_sim[bad_index].head(5)
        # 1. Safely convert to float, turning unparseable text into NaN
        s = pd.to_numeric(df_sim[col], errors='coerce')
        # 2. Fill missing/NaN values with 0.0 so addition doesn't fail
        s = s.fillna(0.0)
        # 3. Add to total distance
        df_sim['distance'] += s * float(weight)
    df_sim['weight'] = (1 - df_sim['distance']) * 100
    return df_sim

# Get the count of the total sample
count_sample = len(df_sample.index)
# Make raw distance dataframe
# Make weighted similarities (similarity = 1 - distance)
df_sim = make_weighted_similaries(df_raw_dist, distance_weights=distance_weights)
# df_sim.head(5)


73153 of 73153 with null results for pred__24_object_type__v_dist_normal
73153 of 73153 with null results for pred__24_fabric_category__v_dist_normal


In [10]:
def make_reduced_dist_df(df_raw_dist, threshold=0.75):
    """Makes a dataframe with distances, but eliminating unused columns"""
    df_reduced_dist =  df_raw_dist.copy()
    cols = df_reduced_dist.columns.tolist()
    act_distance_weights = {k:v for k, v in distance_weights.items() if k in cols}
    drop_cols = []
    ok_cols = []
    for col, weight in act_distance_weights.items():
        col_index = ~df_raw_dist[col].isnull()
        if df_raw_dist[col_index].empty:
            # we have a column with no distance values, so don't make it an option for changing weights
            drop_cols.append(col)
            continue
        zero_index = col_index & (df_raw_dist[col] > 0)
        if df_raw_dist[zero_index].empty:
            # we have a column with no differences in distances, so don't make it an option for changing weights
            drop_cols.append(col)
            continue
        ok_cols.append(col)
    use_cols = ['source', 'target'] + ok_cols
    df_reduced_dist = df_raw_dist[use_cols].copy()
    df_reduced_dist.rename(columns={'source': 'from', 'target': 'to'}, inplace=True)
    use_distance_weights = {k:v for k, v in distance_weights.items() if k in df_reduced_dist.columns.tolist()}
    return df_reduced_dist, use_distance_weights

df_reduced_dist, use_distance_weights = make_reduced_dist_df(df_raw_dist)
json_edge_distances = df_reduced_dist.to_json(orient='records')
distance_weights_labeled = {k: {'weight': w, 'label': distance_col_labels.get(k)} for k, w in use_distance_weights.items()}
json_distance_weights_labeled = json.dumps(distance_weights_labeled)

    

In [11]:

def percentile_df_sim_weights(df_sim):
    """Makes percentiles for df_sim weights for each source"""
    df_sim['source_max_weight'] = float(0.0)
    df_sim['source_min_weight'] = float(0.0)
    quantiles = [0.98, 0.9, .75, 0.5, .25, 0.1,]
    q_cols = {f'source_{int(q * 100)}_weight': q for q in quantiles}
    for col, _ in q_cols.items():
        df_sim[col] = float(0.0)
    for source in df_sim['source'].unique():
        act_index = df_sim['source'] == source
        df_sim.loc[act_index, 'source_max_weight'] = df_sim[act_index]['weight'].max()
        df_sim.loc[act_index, 'source_min_weight'] = df_sim[act_index]['weight'].min()
        for col, q in q_cols.items():
            df_sim.loc[act_index, col] = df_sim[act_index]['weight'].quantile(q)
    return df_sim

def select_min_weights_by_col(df_sim, col_criteria):
    df_sim['use_weight'] = False
    for i, row in df_sim.iterrows():
        df_sim.at[i, 'use_weight'] = (row['weight'] >= row[col_criteria])
    return df_sim

df_sim = percentile_df_sim_weights(df_sim)
# df_sim = select_min_weights_by_col(df_sim, col_criteria='source_90_weight')
df_sim = select_min_weights_by_col(df_sim, col_criteria='source_max_weight')


In [12]:
#act_index = df_sim['weight'] >= df_sim['source_98_weight']
q_top = df_sim['weight'].quantile(0.99)
act_index = df_sim['weight'] >= df_sim['source_max_weight']
act_index |= df_sim['weight'] >= q_top
print(f'Make force directed graph with {len(df_sim[act_index].index)} edges')

Make force directed graph with 944 edges


In [13]:
# Make options based on the size of the total number of nodes

iterations = 750 - round((count_sample * 2.5), 0)
if iterations < 75:
    iterations = 75

iterations = 25
options = (
"""
{
  "physics": {
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -175,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08,
      "damping": 0.6
    },
    "maxVelocity": 50,
    "minVelocity": 0.75,
    "timestep": 0.5,
    "damping": 0.6,
    "stabilization": {
      "enabled": true,
      "iterations": XX,
      "updateInterval": 25,
      "onlyDynamicEdges": false,
      "fit": true
    }
  },
  "edges": {
    "smooth": false
  },
  "interaction": {
    "hideEdgesOnDrag": true
  }
}
"""
)
options = options.replace('"iterations": XX,', f'"iterations": {iterations},')

In [14]:
from pyvis.network import Network
import networkx as nx

# 1. Create the network for the force-directed graph

net = Network(
    height='750px', 
    width='100%', 
    bgcolor='#222222', 
    font_color='white', 
    notebook=True, 
    cdn_resources='in_line'
)

# 2. Set physics options, dymanically scaled with the number of nodes
net.set_options(options)


# 3. Add unique nodes with image properties
all_nodes = df_sim[act_index]['source'].unique().tolist()
all_nodes += df_sim[act_index]['target'].unique().tolist()
unique_nodes = set(all_nodes)
count_nodes = len(unique_nodes)
for uuid_g in unique_nodes:
    uuid = uuid_g.replace('_', '-')
    uuid_index = df_sample['uuid'] == uuid
    label = df_sample[uuid_index]['label'].iloc[0]
    thumb_uri = df_sample[uuid_index]['thumbnail_uri'].iloc[0]
    if str(thumb_uri).startswith('http'):
        net.add_node(
            uuid_g, 
            label=label,
            url=f'https://opencontext.org/subjects/{uuid}',
            shape='image', 
            image=thumb_uri,
            size=25,
        )
    else:
        net.add_node(
            uuid_g, 
            label=label,
            url=f'https://opencontext.org/subjects/{uuid}',
            shape='dot', 
            size=20,
        )

# 4. Add edges
max_len = 100
min_len = 30
max_weight = df_sim[act_index]['weight'].max()
for _, row in df_sim[act_index].iterrows():
    value = (row['weight'] / max_weight) * 15
    if False and row['weight'] >= q_top:
        spring_length = max_len - (row['weight'] / max_weight)  * (max_len - min_len)
        if spring_length < min_len:
            spring_length = min_len
        if spring_length > max_len:
            spring_length = max_len
        net.add_edge(
            row['source'], 
            row['target'], 
            value=value,
            length=spring_length,
            weight=row['weight'],
            physics=(row['weight'] > 0.05),
        )
    else:
        net.add_edge(
            row['source'], 
            row['target'], 
            value=value,
            weight=row['weight'],
        )

# 5. Generate and open the graph
# net.toggle_physics(True)
net.prep_notebook()



In [15]:
# net.show(net_output_html, notebook=False)
net.save_graph(net_output_html)

# Add this AFTER calling net.write_html() or generating your file
with open(net_output_html, "r") as f:
    html = f.read()


add_data = f"""
    buildWeightSliders();
    document.getElementById('edge-status').textContent = '{count_nodes} nodes, ' + edges.length.toLocaleString() + ' edges shown';
"""


# Inject JS event listener to freeze physics on load, add onclick event
add_script = """

network = new vis.Network(container, data, options);

savedPhysicsOptions = JSON.parse(JSON.stringify(options.physics));

document.getElementById('physics-toggle-btn').addEventListener('click', togglePhysicsPause);

// network.clusterByHubsize(4);

network.once("stabilized", function() {
    network.setOptions({ physics: false });
});
network.on("click", function(params) {
    if (params.nodes.length > 0) {
        var nodeId = params.nodes[0];
        var nodeData = nodes.get(nodeId);
        if (network.isCluster(nodeId)) {
            network.openCluster(nodeId);
        }
        if (nodeData && nodeData.url) {
            window.open(nodeData.url, '_blank'); // Opens in a new tab
        }
    }
});

"""

head_html = f"""
<meta charset="utf-8"><title>{TITLE}</title>
<link href="https://fonts.googleapis.com/css2?family=Open+Sans:wght@300;400;600&display=swap" rel="stylesheet">
<meta property="og:image" content="https://opencontext.org/static/oc/images/index/oc-blue-square-logo.png" />
<meta property="og:image:secure_url" content="https://opencontext.org/static/oc/images/index/oc-blue-square-logo.png" />
<meta property="og:image:alt" content="The Open Context logo" />
<meta property="og:description" content="{TITLE}: interactive visualization of data hosted by Open Context" />
<meta name="description"content="{TITLE}: interactive visualization of data hosted by Open Context" />
<link rel="shortcut icon" href="https://opencontext.org/static/oc/images/oc-favicon.ico" />
"""

html = html.replace("network = new vis.Network(container, data, options);", (add_data + add_script), 1)
html = html.replace('<meta charset="utf-8">', f'<meta charset="utf-8">\n{head_html}', 1)


In [16]:
control_css = """
<style type="text/css">

    .network-wrapper {
         position: relative;
         width: 100%;
         font-family: 'Open Sans', sans-serif;
    }

    #control-panel {
         position: absolute;
         top: 10px;
         right: 10px;
         z-index: 20;
         display: flex;
         gap: 8px;
         padding: 8px 10px;
         background: rgba(30, 30, 30, 0.85);
         border: 1px solid rgba(255, 255, 255, 0.15);
         border-radius: 6px;
         box-shadow: 0 2px 8px rgba(0, 0, 0, 0.35);
         font-family: sans-serif;
         font-size: 13px;
     }

     #control-panel button {
         padding: 5px 12px;
         border: 1px solid rgba(255, 255, 255, 0.25);
         border-radius: 4px;
         background: rgba(255, 255, 255, 0.1);
         color: #f0f0f0;
         cursor: pointer;
     }

     #control-panel button:hover {
         background: rgba(255, 255, 255, 0.2);
     }

     #control-panel button:active {
         background: rgba(255, 255, 255, 0.28);
     }

      #weight-panel {
         position: absolute;
         top: 10px;
         left: 10px;
         z-index: 20;
         width: 260px;
         max-height: calc(100vh - 40px);
         overflow-y: auto;
         padding: 10px 12px;
         background: rgba(30, 30, 30, 0.9);
         border: 1px solid rgba(255, 255, 255, 0.15);
         border-radius: 6px;
         box-shadow: 0 2px 8px rgba(0, 0, 0, 0.35);
         font-family: sans-serif;
         font-size: 12px;
         color: #f0f0f0;
     }

     #weight-panel-header {
         display: flex;
         justify-content: space-between;
         align-items: center;
         gap: 8px;
         margin-bottom: 10px;
     }

     #weight-panel.weight-panel-collapsed #weight-panel-header {
         margin-bottom: 0;
     }

     #weight-panel.weight-panel-collapsed #weight-panel-body {
         display: none;
     }

     #weight-panel.weight-panel-collapsed {
         width: auto;
     }

     #weight-panel h3 {
         margin: 0;
         font-size: 13px;
         font-weight: 600;
     }

     #weight-panel-toggle {
         padding: 2px 8px;
         border: 1px solid rgba(255, 255, 255, 0.25);
         border-radius: 4px;
         background: rgba(255, 255, 255, 0.1);
         color: #f0f0f0;
         cursor: pointer;
         font-size: 11px;
         line-height: 1.4;
         white-space: nowrap;
     }

     #weight-panel-toggle:hover {
         background: rgba(255, 255, 255, 0.2);
     }

     .weight-slider-row {
         margin-bottom: 10px;
     }

     .weight-slider-row label {
         display: flex;
         justify-content: space-between;
         align-items: baseline;
         margin-bottom: 4px;
     }

     .weight-slider-row input[type="range"] {
         width: 100%;
     }

     #edge-status {
         margin-top: 8px;
         font-size: 11px;
         color: rgba(255, 255, 255, 0.7);
     }

    .logo {
        height: 1.2em;
        margin-left: 0.1em;
        margin-right: 0.25em;
    }
</style>
"""

title_control_panel = f"""
{control_css}
    <h1 class="main"><img class="logo" src="https://opencontext.org/static/oc/images/about/oc-reduced-logo-sm.png" alt="Open Context Logo" />{TITLE}</h1>
    <div class="network-wrapper">
        <div id="weight-panel">
            <div id="weight-panel-header">
                <h3>Similarity Weights</h3>
                <button id="weight-panel-toggle" type="button" aria-expanded="true" title="Hide weight panel">Hide</button>
            </div>
            <div id="weight-panel-body">
                <div id="weight-sliders"></div>
                <div id="edge-status"></div>
            </div>
        </div>
        <div id="mynetwork" class="card-body"></div>
        <div id="control-panel">
            <button id="physics-toggle-btn" type="button">Pause Motion</button>
        </div>
    </div>
"""

html = html.replace('<div id="mynetwork" class="card-body"></div>', title_control_panel, 1)

In [17]:
add_motion_toggle = """

              var physicsPaused = false;
              var physicsSlowingDown = false;
              var physicsSlowdownFrame = null;
              var savedPhysicsOptions = null;

              function easePhysicsToHalt(durationMs) {

                  savedPhysicsOptions.damping ??= 0.6;
                  
                  if (physicsSlowdownFrame !== null) {
                      cancelAnimationFrame(physicsSlowdownFrame);
                      physicsSlowdownFrame = null;
                  }

                  physicsSlowingDown = true;
                  document.getElementById('physics-toggle-btn').textContent = 'Pausing...';

                  var startDamping = savedPhysicsOptions.damping;
                  var startTimestep = savedPhysicsOptions.timestep;
                  var startMinVelocity = savedPhysicsOptions.minVelocity;
                  var startTime = performance.now();

                  function step(now) {
                      var t = Math.min(1, (now - startTime) / durationMs);
                      var eased = t * t;

                      network.setOptions({
                          physics: {
                              damping: startDamping + (0.98 - startDamping) * eased,
                              timestep: startTimestep * (1 - eased),
                              minVelocity: startMinVelocity + (50 - startMinVelocity) * eased
                          }
                      });

                      if (t < 1) {
                          physicsSlowdownFrame = requestAnimationFrame(step);
                      } else {
                          physicsSlowdownFrame = null;
                          physicsSlowingDown = false;
                          network.setOptions({ physics: false });
                          physicsPaused = true;
                          document.getElementById('physics-toggle-btn').textContent = 'Resume Motion';
                      }
                  }

                  physicsSlowdownFrame = requestAnimationFrame(step);
              }

              function resumePhysics() {
                  if (physicsSlowdownFrame !== null) {
                      cancelAnimationFrame(physicsSlowdownFrame);
                      physicsSlowdownFrame = null;
                  }

                  physicsSlowingDown = false;
                  network.setOptions({ physics: savedPhysicsOptions });
                  physicsPaused = false;
                  document.getElementById('physics-toggle-btn').textContent = 'Pause Motion';
              }

              function togglePhysicsPause() {
                  if (physicsPaused || physicsSlowingDown) {
                      resumePhysics();
                  } else {
                      easePhysicsToHalt(1000);
                  }
              }

"""

js_config_distance_weights = f"""
    var all_distances;
    var distance_weights_labeled;
    all_distances = new vis.DataSet({json_edge_distances});
    distance_weights_labeled = JSON.parse('{json_distance_weights_labeled}');
    const count_nodes = {count_nodes};
"""

js_weights = """
            var weightRegenerationTimeout = null;
              var dimensionKeys = [];

              function quantile(sortedValues, q) {
                  if (!sortedValues.length) {
                      return 0;
                  }
                  var pos = (sortedValues.length - 1) * q;
                  var base = Math.floor(pos);
                  var rest = pos - base;
                  if (sortedValues[base + 1] !== undefined) {
                      return sortedValues[base] + rest * (sortedValues[base + 1] - sortedValues[base]);
                  }
                  return sortedValues[base];
              }

              function getActiveDistanceWeights() {
                  var weights = {};
                  for (var i = 0; i < dimensionKeys.length; i++) {
                      var key = dimensionKeys[i];
                      weights[key] = distance_weights_labeled[key].weight;
                  }
                  return weights;
              }

              function computeWeightedSimilarity(record, weights) {
                  var distance = 0;
                  for (var key in weights) {
                      if (weights.hasOwnProperty(key) && record[key] != null) {
                          distance += record[key] * weights[key];
                      }
                  }
                  return {
                      distance: distance,
                      weight: (1 - distance) * 100
                  };
              }

              function generateEdgesFromDistances() {
                  var weights = getActiveDistanceWeights();
                  var records = all_distances.get();
                  var computed = new Array(records.length);
                  var sourceMax = {};

                  for (var i = 0; i < records.length; i++) {
                      var record = records[i];
                      var metrics = computeWeightedSimilarity(record, weights);
                      computed[i] = {
                          from: record.from,
                          to: record.to,
                          weight: metrics.weight
                      };
                      var fromId = record.from;
                      if (sourceMax[fromId] === undefined || metrics.weight > sourceMax[fromId]) {
                          sourceMax[fromId] = metrics.weight;
                      }
                  }

                  var allWeights = computed.map(function(item) { return item.weight; });
                  allWeights.sort(function(a, b) { return a - b; });
                  var qTop = quantile(allWeights, 0.99);

                  var selected = [];
                  for (var j = 0; j < computed.length; j++) {
                      var item = computed[j];
                      if (item.weight >= sourceMax[item.from] || item.weight >= qTop) {
                          selected.push(item);
                      }
                  }

                  if (!selected.length) {
                      return [];
                  }

                  var maxWeight = selected[0].weight;
                  for (var k = 1; k < selected.length; k++) {
                      if (selected[k].weight > maxWeight) {
                          maxWeight = selected[k].weight;
                      }
                  }

                  return selected.map(function(item) {
                      return {
                          from: item.from,
                          to: item.to,
                          value: maxWeight > 0 ? (item.weight / maxWeight) * 15 : 0,
                          weight: item.weight
                      };
                  });
              }

              function updateGraphEdges(newEdgeList) {
                  var edgeIds = edges.getIds();
                  if (edgeIds.length) {
                      console.log('remove old edges: ' + edgeIds.length);
                      edges.remove(edgeIds);
                  }
                  edges.add(newEdgeList);
                  allEdges = edges.get({ returnType: "Object" });

                  var statusEl = document.getElementById('edge-status');
                  if (statusEl) {
                      statusEl.textContent = count_nodes.toLocaleString() + ' nodes, ' + newEdgeList.length.toLocaleString() + ' edges shown';
                  }

                  if (network) {
                      network.redraw();
                  }
                  if (false) {
                      network.redraw();
                      if (physicsPaused) {
                          resumePhysics();
                          network.stabilize(25);
                          if (false) {
                              network.once("stabilized", function() {
                                  network.setOptions({ physics: false });
                              });
                          }
                      } else {
                          network.stabilize(25);
                      }
                  }
              }

              function regenerateEdgesFromWeights() {
                  var statusEl = document.getElementById('edge-status');
                  if (statusEl) {
                      statusEl.textContent = 'Regenerating edges...';
                  }

                  window.setTimeout(function() {
                      var newEdges = generateEdgesFromDistances();
                      updateGraphEdges(newEdges);
                  }, 0);
              }

              function scheduleEdgeRegeneration() {
                  if (weightRegenerationTimeout !== null) {
                      window.clearTimeout(weightRegenerationTimeout);
                  }
                  weightRegenerationTimeout = window.setTimeout(function() {
                      weightRegenerationTimeout = null;
                      regenerateEdgesFromWeights();
                  }, 300);
              }

              function initWeightPanelToggle() {
                  var panel = document.getElementById('weight-panel');
                  var toggleBtn = document.getElementById('weight-panel-toggle');
                  if (!panel || !toggleBtn || toggleBtn.dataset.initialized === 'true') {
                      return;
                  }

                  toggleBtn.dataset.initialized = 'true';
                  toggleBtn.addEventListener('click', function() {
                      var collapsed = panel.classList.toggle('weight-panel-collapsed');
                      toggleBtn.textContent = collapsed ? 'Show' : 'Hide';
                      toggleBtn.setAttribute('aria-expanded', collapsed ? 'false' : 'true');
                      toggleBtn.title = collapsed ? 'Show weight panel' : 'Hide weight panel';
                  });
              }

              function buildWeightSliders() {
                  initWeightPanelToggle();

                  var container = document.getElementById('weight-sliders');
                  if (!container) {
                      return;
                  }

                  container.innerHTML = '';
                  dimensionKeys = Object.keys(distance_weights_labeled);

                  dimensionKeys.forEach(function(key) {
                      var config = distance_weights_labeled[key];
                      var row = document.createElement('div');
                      row.className = 'weight-slider-row';

                      var label = document.createElement('label');
                      label.setAttribute('for', 'weight-slider-' + key);
                      label.innerHTML = '<span>' + config.label + '</span><span id="weight-value-' + key + '">' + config.weight.toFixed(2) + '</span>';

                      var slider = document.createElement('input');
                      slider.type = 'range';
                      slider.min = '0';
                      slider.max = '1';
                      slider.step = '0.01';
                      slider.value = String(config.weight);
                      slider.id = 'weight-slider-' + key;

                      slider.addEventListener('input', function() {
                          var value = parseFloat(slider.value);
                          distance_weights_labeled[key].weight = value;
                          document.getElementById('weight-value-' + key).textContent = value.toFixed(2);
                          scheduleEdgeRegeneration();
                      });

                      row.appendChild(label);
                      row.appendChild(slider);
                      container.appendChild(row);
                  });
              }

"""

html = html.replace(
    '// This method is responsible for drawing the graph, returns the drawn network', 
    f'{add_motion_toggle}\n{js_config_distance_weights}\n{js_weights}\n// This method is responsible for drawing the graph, returns the drawn network', 
    1
)

In [18]:
with open(net_output_html, "w") as f:
    f.write(html)

from IPython.display import IFrame
from IPython.display import display, HTML

# HTML(filename=net_output_html)